# 第27课 · 概率基础 — 事件、条件概率（Conditional Probability）、独立性（Statistical Independence）与大数定律（Law of Large Numbers，LLN）

**目标**：用 numpy 模拟随机事件，观察「样本（Sample）多了，频率（Relative Frequency）趋近概率」。

**为什么对 Aurora 重要**：`np.random.default_rng(seed)` 贯穿训练全流程——batch 采样（Sampling）用它洗牌、dropout 用它生成掩码（Mask）、数据增强（Data Augmentation）用它选变换参数；seed 固定后结果可复现，调参和 debug 才有意义。

← **上一课**　[L26 · 微积分可视化](../3_calculus/L26_visual_calculus.ipynb)

> 上节课学习了 **微积分可视化**：切线、等高线与梯度下降轨迹动态演示。  
> 本课将探讨 **概率基础**。

## 本课剧情：为什么骰子"说谎"但概率不说谎？

掷一次骰子，你不知道会出几。掷一万次，出现"6"的频率会稳定在 1/6 ≈ 16.67%——误差越来越小。

这就是**大数定律（Law of Large Numbers，LLN）**：

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{n \to \infty} \mathbb{E}[X]$$

样本均值随 n 增大趋近总体期望。随机性不消失，但它的**平均效果可以预测**。

这和 Aurora 有什么关系？

1. **模型训练**：Loss 是一个 batch 里所有样本损失的平均值。batch 越大，这个平均越接近"真实期望损失"——这正是为什么大 batch 训练更稳定。
2. **数据增强**：用不同 seed 对同一音频做随机 pitch/noise 变换，多次的期望效果等价于对整个变换分布积分。
3. **评估指标**：测试集的 WER/CER 是"随机采样"的频率估计；测试集越大，估计越准确。

本课实现 `estimate_prob_six(n, seed)`，验证 LLN；并理解条件概率和贝叶斯定理——这是 L30 交叉熵损失的前置知识。

## 大数定律（Law of Large Numbers）正式表述

设 $X_1, X_2, \ldots, X_n$ 是独立同分布（i.i.d.）随机变量，期望为 $\mu = E[X]$，则样本均值

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i$$

**几乎必然收敛**到 $\mu$（强大数定律）：

$$\bar{X}_n \xrightarrow{\text{a.s.}} \mu \quad (n \to \infty)$$

下面的代码实验是对这个收敛过程的**可视化**：用掷骰子（$\mu = 3.5$）和出现6的频率（$\mu = 1/6$）演示 $\bar{X}_n$ 如何随 $n$ 增大而稳定。


## 1. 大数定律的实验验证

**为什么要固定 seed？**

`np.random.default_rng(seed)` 创建独立的随机状态对象（PCG64 算法），每次用相同 seed 得到完全相同的随机序列。这保证了：
- **实验可复现**：同学的代码和你的输出一致
- **调试可追踪**：出了问题知道是哪一个随机序列

**LLN 的具体数字**：

| n（掷骰次数）| 出现"6"的频率 | 误差量级 |
|---|---|---|
| 10 | 波动剧烈，±0.3 | O(1/√10) ≈ 0.32 |
| 1,000 | 约 ±0.03 | O(1/√1000) ≈ 0.032 |
| 200,000 | 约 ±0.001 | O(1/√200000) ≈ 0.002 |

误差收缩速度：O(1/√n)——所以想把精度提高 10 倍，需要 100 倍的样本。

> **条件概率定义**：P(A|B) = P(A∩B) / P(B)  
> **独立性定义**：若 P(A∩B) = P(A)·P(B)，则 A, B 独立。  
> **贝叶斯定理**：P(A|B) = P(B|A)·P(A) / P(B)——这正是 L30 交叉熵中 "用模型 q 近似真实 p" 的概率论基础。

## 实验入口：概率量如何从样本里出现

注意 `size` 和 `seed` 两个参数：`size` 决定估计精度，`seed` 决定结果是否可复现。运行下面的格，观察 `flips.mean()` 是否接近 0.5。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
flips = rng.integers(0, 2, size=1000)   # 0/1 当正反面
print('1000 次抛硬币，正面比例 ≈', flips.mean(), ' (应接近 0.5)')

## 动手观察：小样本 vs 大样本

下面对比 `n=10`、`100`、`10000` 时掷骰子的频率估计，观察样本量增大时估计值如何向理论值 0.167 靠拢。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    print(f'n={n:5d}  估计 P(掷到6) = {estimate:.3f}')


## 代码实验：多次重复同一个随机实验

用 5 个不同 `seed` 重复同一个 `n` 的实验：`n=2000` 时各次估计值聚集在 0.167 附近，`n=20` 时则明显分散。

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        rolls = rng.integers(1, 7, size=n)
        estimates.append(np.mean(rolls == 6))
    print(f'n={n:4d} ->', np.round(estimates, 3), '平均=', round(float(np.mean(estimates)), 3))


## 2. ✏️ 实现 `estimate_prob_six(n, seed=0)`

模拟掷 n 次骰子(1~6)，返回出现 6 的频率。

**推理路线**：
1. 创建独立随机状态：`rng = np.random.default_rng(seed)`，不污染全局随机序列。
2. 一次性生成 n 次投掷：`rolls = rng.integers(1, 7, size=n)`，区间是 `[1, 7)`，即 1~6。
3. 统计 6 的频率：`np.mean(rolls == 6)` 等价于 `sum(rolls == 6) / n`，返回 0~1 的浮点数。

**参考输入输出**：n=6, seed=0 → 可能得到 `[4,5,6,6,1,3]`，频率=2/6≈0.333；n=10000, seed=0 → 频率会收敛到约 0.166。

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `estimate_prob_six` 前明确三件事：
- 输入：`n`（掷骰次数）和 `seed`（随机种子，默认 0）
- 关键步骤：用 `rng.integers(1, 7, size=n)` 生成样本，对 `sample == 6` 取均值（Mean）
- 返回：一个浮点数，表示 6 出现的频率，`n=200000` 时应在 0.165–0.169 之间

In [ ]:
def estimate_prob_six(n, seed=0):
    # ✏️ TODO: 返回掷 n 次骰子中 6 出现的频率
    raise NotImplementedError("请实现 estimate_prob_six：用 rng.integers(1,7,size=n) 生成样本，返回 (sample==6).mean()")


In [ ]:
try:
    p = estimate_prob_six(200_000)
    print("P(掷出6) ≈", round(p, 4), " (真值 1/6 ≈ 0.1667)")
    assert abs(p - 1/6) < 0.01
    print("✅ 通过：样本够多，频率逼近真实概率。")
except NotImplementedError as e:
    print(f"⚠️  请先完成练习：{e}")


**🔗 Aurora 连接**：`src/aurora/audio/` 中的数据增强管道和训练循环都通过 `np.random.default_rng(seed)` 生成 batch 洗牌索引和 dropout 掩码。固定 seed 后，两次运行产生完全一致的 batch 序列——这是调参和 debug 时对照实验有意义的前提。

## 参数实验：只改一个旋钮

固定 `seed=7`，把 `n` 从 10 增加到 10000，每次打印频率。观察 n=100 时波动约 ±0.05，n=10000 时波动约 ±0.005（标准差（Standard Deviation，SD） ∝ 1/√n）。

In [ ]:
for n in [20, 100, 1000, 10000]:
    rng = np.random.default_rng(7)
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    error = abs(estimate - 1/6)
    print(f'n={n:5d} | 估计={estimate:.4f} | 离理论值差={error:.4f}')


## 3. 条件概率（Conditional Probability）与独立性（Statistical Independence）

**条件概率**定义：已知事件 B 发生，A 发生的概率：

```
P(A | B) = P(A ∩ B) / P(B)
```

**独立性**：若 `P(A | B) = P(A)`（知道 B 不改变 A 的概率），则 A 与 B 独立。
等价条件：`P(A ∩ B) = P(A) · P(B)`。

**Aurora 连接**：dropout 在每一步随机屏蔽神经元，各神经元的屏蔽事件被设计为独立（mask = 独立 Bernoulli 变量），这样可以用乘法法则分析整个网络的期望输出。

In [ ]:
import numpy as np

# 模拟：掷两枚骰子，验证条件概率
rng = np.random.default_rng(0)
n = 200_000
d1 = rng.integers(1, 7, n)
d2 = rng.integers(1, 7, n)

# A: d1=6  B: d1+d2>9
A = d1 == 6
B = (d1 + d2) > 9

p_A = A.mean()
p_B = B.mean()
p_A_given_B = (A & B).sum() / B.sum()

print(f'P(d1=6)            = {p_A:.4f}  (理论 1/6 ≈ 0.1667)')
print(f'P(d1+d2>9)         = {p_B:.4f}')
print(f'P(d1=6 | d1+d2>9) = {p_A_given_B:.4f}  (应高于 P(d1=6))')

assert p_A_given_B > p_A, '条件概率应高于先验概率'
print('✅ P(A|B) > P(A)：知道 B 发生提升了 A 的概率')


In [ ]:
# ✅ 独立性数值验证：两枚独立骰子，验证乘法法则 P(A∩B) ≈ P(A)·P(B)
rng2 = np.random.default_rng(1)
n = 200_000
die1 = rng2.integers(1, 7, n)   # 第一枚骰子（独立）
die2 = rng2.integers(1, 7, n)   # 第二枚骰子（独立）

A_indep = die1 == 6              # 事件 A：第一枚骰子出6
B_indep = die2 == 1              # 事件 B：第二枚骰子出1（独立于第一枚）

p_A_i = A_indep.mean()
p_B_i = B_indep.mean()
p_AB_i = (A_indep & B_indep).mean()
product = p_A_i * p_B_i

print(f'P(A)           = {p_A_i:.4f}  (理论 1/6 ≈ 0.1667)')
print(f'P(B)           = {p_B_i:.4f}  (理论 1/6 ≈ 0.1667)')
print(f'P(A∩B)         = {p_AB_i:.5f}')
print(f'P(A)·P(B)      = {product:.5f}')
print(f'差值 |P(A∩B) - P(A)·P(B)| = {abs(p_AB_i - product):.5f}')

assert abs(p_AB_i - product) < 0.001, (
    f'乘法法则验证失败：|{p_AB_i:.5f} - {product:.5f}| 应 < 0.001'
)
print('✅ 乘法法则成立：两枚骰子的结果统计独立。')


## 4. 贝叶斯定理（Bayes Theorem）

由条件概率定义推导：

```
P(A | B) = P(B | A) * P(A) / P(B)
```

- `P(A)` — **先验概率（Prior）**：观测前对 A 的估计
- `P(B | A)` — **似然（Likelihood）**：若 A 为真，B 出现的概率
- `P(A | B)` — **后验概率（Posterior）**：观测到 B 后更新的估计

**垃圾邮件示例**：
- `P(spam)=0.3`, `P(含free|spam)=0.9`, `P(含free|ham)=0.01`
- `P(spam|含free) = 0.9*0.3 / (0.9*0.3 + 0.01*0.7) ≈ 0.975`
- 先验 30% → 后验 97.5%：观测到证据后信念大幅更新

In [ ]:
# 贝叶斯定理数值验证
p_spam = 0.30
p_free_given_spam = 0.90
p_free_given_ham  = 0.01

p_free = p_free_given_spam * p_spam + p_free_given_ham * (1 - p_spam)
p_spam_given_free = p_free_given_spam * p_spam / p_free

print(f'P(spam)                = {p_spam:.3f}')
print(f'P(free | spam)         = {p_free_given_spam:.3f}')
print(f'P(free)                = {p_free:.3f}')
print(f'P(spam | free)         = {p_spam_given_free:.3f}')

assert abs(p_spam_given_free - 0.975) < 0.002
print('✅ 贝叶斯定理：先验 0.30 → 后验 0.975')


## 本课收束

现在可以用 `estimate_prob_six(n, seed)` 返回掷骰频率，`n=200000` 时估计值稳定在 0.1667 以内。Aurora 训练循环中的 batch 洗牌和 dropout mask 生成都依赖相同的 `default_rng` 接口，固定 seed 是让实验可复现的最小代价。下一节将用均值和方差刻画样本分布的形状，今天的 `np.mean(sample == 6)` 是样本均值的特例。频率随 `n` 增大收敛的速率是 1/√n，这也是下节引入标准差时的核心关系。

In [ ]:
# 小检查：同一个实验，样本量越大越稳定
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.integers(1, 7, size=n)
    estimate = np.mean(samples == 6)
    print(f'n={n:4d} -> P(6)≈{estimate:.3f}')


## ✏️ 白板挑战：概率手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：P(掷出6) = ?（精确分数）。掷 1000 次骰子，期望出现"6"多少次？

**问 2**：两枚骰子 A、B 独立投掷。P(A=6) = 1/6，P(B=6) = 1/6。  
P(A=6 且 B=6) = ?  
独立性判据：P(A∩B) = P(A)·P(B) 成立吗？

**问 3**：已知：
- P(垃圾邮件) = 0.30
- P("free"出现 | 垃圾邮件) = 0.90
- P("free"出现 | 非垃圾) = 0.01

用贝叶斯定理，计算 P(垃圾邮件 | 邮件含"free") = ?

**问 4**：样本量 n 从 100 增大到 10000，频率估计的误差量级从 O(?) 变为 O(?)？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：P(6)=1/6, 1000次期望=1000/6≈166.7次
p_six = 1/6
expected_count = 1000 * p_six
assert np.isclose(p_six, 1/6, atol=1e-12)
assert np.isclose(expected_count, 1000/6, atol=1e-10)
print(f"Q1 ✅  P(6)={p_six:.6f}=1/6，1000次期望={expected_count:.1f}次")

# 数值验证LLN
try:
    p_est = estimate_prob_six(200_000, seed=42)
    assert abs(p_est - 1/6) < 0.005, f"估计误差过大: {abs(p_est-1/6):.4f}"
    print(f"     LLN验证: n=200000时估计={p_est:.5f}，误差={abs(p_est-1/6):.5f}")
except NotImplementedError:
    print(f"     (estimate_prob_six 待实现)")

# 问2：独立性 P(A∩B) = P(A)·P(B) = 1/36
p_ab = p_six * p_six
assert np.isclose(p_ab, 1/36, atol=1e-12)
# 数值验证
rng = np.random.default_rng(99)
dice1 = rng.integers(1, 7, 200_000)
dice2 = rng.integers(1, 7, 200_000)
p_both_6 = np.mean((dice1 == 6) & (dice2 == 6))
assert abs(p_both_6 - 1/36) < 0.002
print(f"Q2 ✅  P(A=6∩B=6)=1/36={1/36:.5f}，数值={p_both_6:.5f}，独立性成立")

# 问3：贝叶斯定理
p_spam = 0.30
p_free_spam = 0.90
p_free_ham  = 0.01
p_free = p_free_spam * p_spam + p_free_ham * (1 - p_spam)
p_spam_given_free = p_free_spam * p_spam / p_free
assert np.isclose(p_free, 0.277, atol=1e-10)
assert np.isclose(p_spam_given_free, 0.9 * 0.3 / 0.277, atol=1e-10)
print(f"Q3 ✅  P('free')={p_free:.4f}，P(垃圾|free)={p_spam_given_free:.4f}")

# 问4：误差量级 O(1/√n)
n1, n2 = 100, 10000
err1 = 1 / np.sqrt(n1)
err2 = 1 / np.sqrt(n2)
ratio = err1 / err2
assert np.isclose(ratio, 10.0, atol=1e-10)
print(f"Q4 ✅  n=100→O(1/√100)={err1:.3f}，n=10000→O(1/√10000)={err2:.3f}，误差缩小{ratio:.0f}倍")
print("\n🎉 概率白板挑战通过！LLN、条件概率、贝叶斯定理三大工具已内化。")

In [ ]:
# ✏️ 本课自评
l27_review = {
    "lln_statement":              None,  # 能陈述大数定律（X̄ₙ → E[X]）？True/False
    "estimate_prob_six_done":     None,  # estimate_prob_six 实现并通过断言？True/False
    "conditional_probability":    None,  # 理解 P(A|B)=P(A∩B)/P(B)？True/False
    "bayes_theorem_computed":     None,  # 能用贝叶斯定理手算后验概率？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l27_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l27_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L27 全部通关！进入 L28：均值方差标准化')

---

→ **下一课**　[L28 · 均值方差标准化](L28_descriptive_stats.ipynb)

> 下节课将学习 **均值方差标准化**：描述性统计、z-score 标准化与分布比较。